# Conditional Logic Testing

This notebook tests:
1. Survey logic engine
2. Skip patterns and conditional questions
3. Dependency resolution
4. Path tracing

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.logic.conditional_logic import (
    create_standard_survey_logic,
    ConditionalRule,
    Operator
)

import warnings
warnings.filterwarnings('ignore')

## 1. Initialize Survey Logic

In [2]:
logic = create_standard_survey_logic()

print("Survey Logic Initialized")
print(f"Total conditional rules: {len(logic.rules)}")
print(f"Question order: {len(logic.question_order)} questions")
print(f"Screening rules: {len(logic.screening_rules)}")

Survey Logic Initialized
Total conditional rules: 5
Question order: 20 questions
Screening rules: 2


## 2. Test Individual Rules

In [3]:
# Test B21 (Barriers) - only if low purchase intent
print("Testing B21 (Barriers) conditional logic")
print("="*80)
print("Rule: Ask B21 if B2 ∈ {4, 5} (low purchase intent)\n")

test_cases = [
    ({'B2': 1}, False, "B2=1 (Definitely would) → Don't ask barriers"),
    ({'B2': 2}, False, "B2=2 (Probably would) → Don't ask barriers"),
    ({'B2': 3}, False, "B2=3 (Might or might not) → Don't ask barriers"),
    ({'B2': 4}, True, "B2=4 (Probably would not) → Ask barriers"),
    ({'B2': 5}, True, "B2=5 (Definitely would not) → Ask barriers"),
    ({}, False, "B2 not answered → Don't ask barriers"),
]

for responses, expected, description in test_cases:
    should_ask = logic.should_ask_question('B21', responses)
    status = "✓" if should_ask == expected else "✗"
    print(f"{status} {description}")
    if should_ask != expected:
        print(f"   ERROR: Expected {expected}, got {should_ask}")

Testing B21 (Barriers) conditional logic
Rule: Ask B21 if B2 ∈ {4, 5} (low purchase intent)

✓ B2=1 (Definitely would) → Don't ask barriers
✓ B2=2 (Probably would) → Don't ask barriers
✓ B2=3 (Might or might not) → Don't ask barriers
✓ B2=4 (Probably would not) → Ask barriers
✓ B2=5 (Definitely would not) → Ask barriers
✓ B2 not answered → Don't ask barriers


In [4]:
# Test B22 (Gift intent) - only if some purchase intent
print("\nTesting B22 (Gift Intent) conditional logic")
print("="*80)
print("Rule: Ask B22 if B2 ∈ {1, 2, 3} (some purchase intent)\n")

test_cases = [
    ({'B2': 1}, True, "B2=1 (Definitely would) → Ask gift intent"),
    ({'B2': 2}, True, "B2=2 (Probably would) → Ask gift intent"),
    ({'B2': 3}, True, "B2=3 (Might or might not) → Ask gift intent"),
    ({'B2': 4}, False, "B2=4 (Probably would not) → Don't ask gift intent"),
    ({'B2': 5}, False, "B2=5 (Definitely would not) → Don't ask gift intent"),
]

for responses, expected, description in test_cases:
    should_ask = logic.should_ask_question('B22', responses)
    status = "✓" if should_ask == expected else "✗"
    print(f"{status} {description}")


Testing B22 (Gift Intent) conditional logic
Rule: Ask B22 if B2 ∈ {1, 2, 3} (some purchase intent)

✓ B2=1 (Definitely would) → Ask gift intent
✓ B2=2 (Probably would) → Ask gift intent
✓ B2=3 (Might or might not) → Ask gift intent
✓ B2=4 (Probably would not) → Don't ask gift intent
✓ B2=5 (Definitely would not) → Don't ask gift intent


## 3. Test Screening Rules

In [5]:
print("Testing Screening Rules")
print("="*80)

screening_test_cases = [
    (
        {'occupation': 'Technology', 'age': 25},
        True,
        "Valid: Tech worker, age 25"
    ),
    (
        {'occupation': 'Marketing/Market Research', 'age': 30},
        False,
        "Screened: Marketing occupation"
    ),
    (
        {'occupation': 'Advertising/PR', 'age': 28},
        False,
        "Screened: Advertising occupation"
    ),
    (
        {'occupation': 'Education', 'age': 17},
        False,
        "Screened: Age too young (17)"
    ),
    (
        {'occupation': 'Retired', 'age': 76},
        False,
        "Screened: Age too old (76)"
    ),
    (
        {'occupation': 'Healthcare', 'age': 45},
        True,
        "Valid: Healthcare, age 45"
    ),
]

for persona_attrs, should_pass, description in screening_test_cases:
    passes, reason = logic.apply_screening(persona_attrs)
    status = "✓" if passes == should_pass else "✗"
    print(f"{status} {description}")
    if not passes:
        print(f"   Reason: {reason}")
    if passes != should_pass:
        print(f"   ERROR: Expected {should_pass}, got {passes}")

Testing Screening Rules
✓ Valid: Tech worker, age 25
✓ Screened: Marketing occupation
   Reason: Works in excluded occupation
✓ Screened: Advertising occupation
   Reason: Works in excluded occupation
✓ Screened: Age too young (17)
   Reason: Age out of range
✓ Screened: Age too old (76)
   Reason: Age out of range
✓ Valid: Healthcare, age 45


## 4. Test Question Flow

In [6]:
# Simulate a respondent's journey
print("Simulating Respondent Journey")
print("="*80)

# Scenario 1: High purchase intent
print("\nScenario 1: High Purchase Intent (B2=1)")
responses = {'B2': 1}  # Definitely would buy

next_questions = logic.get_next_questions(responses, logic.question_order)
print(f"Next questions to ask: {next_questions[:5]}...")

# Check specific conditionals
print(f"   Should ask B21 (Barriers)? {logic.should_ask_question('B21', responses)}")
print(f"   Should ask B22 (Gift)? {logic.should_ask_question('B22', responses)}")

# Scenario 2: Low purchase intent
print("\nScenario 2: Low Purchase Intent (B2=5)")
responses = {'B2': 5}  # Definitely would not buy

print(f"   Should ask B21 (Barriers)? {logic.should_ask_question('B21', responses)}")
print(f"   Should ask B22 (Gift)? {logic.should_ask_question('B22', responses)}")

Simulating Respondent Journey

Scenario 1: High Purchase Intent (B2=1)
Next questions to ask: ['B3', 'B4', 'B6', 'B7', 'B11']...
   Should ask B21 (Barriers)? False
   Should ask B22 (Gift)? True

Scenario 2: Low Purchase Intent (B2=5)
   Should ask B21 (Barriers)? True
   Should ask B22 (Gift)? False


## 5. Full Path Tracing

In [7]:
# Trace complete survey path
print("Tracing Complete Survey Paths")
print("="*80)

persona_scenarios = [
    {
        'name': 'Valid Respondent',
        'attrs': {'occupation': 'Technology', 'age': 30}
    },
    {
        'name': 'Screened (Marketing)',
        'attrs': {'occupation': 'Marketing/Market Research', 'age': 25}
    },
]

for scenario in persona_scenarios:
    print(f"\n{scenario['name']}:")
    
    path = logic.trace_path(scenario['attrs'], logic.question_order)
    
    if not path:
        print("   ⚠️  Screened out - no questions asked")
    else:
        print(f"   Questions to ask: {len(path)}")
        print(f"   Questions: {', '.join(path)}")

Tracing Complete Survey Paths

Valid Respondent:
   Questions to ask: 15
   Questions: B2, B3, B4, B6, B7, B11, B11a, B12, B13, B14, B15, B16, B17, B18, B20

Screened (Marketing):
   ⚠️  Screened out - no questions asked


## 6. Complex Conditional Chains

In [8]:
# Test chained conditionals: B2 → B22 → B23
print("Testing Conditional Chains: B2 → B22 → B23")
print("="*80)

# Chain: High purchase intent → would buy as gift → occasions
print("\nChain 1: B2=1 (high intent) → B22=1 (would gift)")
responses = {'B2': 1, 'B22': 1}
print(f"   Should ask B22? {logic.should_ask_question('B22', {'B2': 1})}")
print(f"   Should ask B23? {logic.should_ask_question('B23', responses)}")

# Chain broken: Low purchase intent → skip gift questions
print("\nChain 2: B2=5 (low intent) → skip B22")
responses = {'B2': 5}
print(f"   Should ask B22? {logic.should_ask_question('B22', responses)}")
print(f"   Should ask B23? {logic.should_ask_question('B23', responses)}")

Testing Conditional Chains: B2 → B22 → B23

Chain 1: B2=1 (high intent) → B22=1 (would gift)
   Should ask B22? True
   Should ask B23? True

Chain 2: B2=5 (low intent) → skip B22
   Should ask B22? False
   Should ask B23? False


## Summary

Conditional logic tested successfully:

✅ **Individual Rules** - All conditional rules evaluate correctly

✅ **Screening** - Occupation and age screening works as expected

✅ **Skip Patterns** - Questions correctly skipped based on dependencies

✅ **Question Flow** - Next questions determined correctly

✅ **Path Tracing** - Complete survey paths can be traced

✅ **Conditional Chains** - Multi-level dependencies work correctly

**Next Steps:**
- Generate full synthetic dataset (notebook 06)
- Verify conditional logic works in production